# 09 — Demo Data Export for Web App

This notebook prepares the final **web-ready demo dataset** for the React/FastAPI application.

It does four important things:

1. Selects a small, clean set of demo customers and articles.
2. Exports home-feed recommendations, similar items, similar users, model metrics, and EDA summary.
3. Generates frontend-friendly image URLs such as `/images/070/0706016001.jpg`.
4. Copies **only the needed demo images** from the original H&M image folder into a small web-ready image folder.

Important: this notebook does **not** copy the full 28GB H&M image folder. It copies only images used by the demo data.


In [1]:
import os
import json
import shutil
import platform
from pathlib import Path
from typing import Dict, List, Optional, Iterable, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)


Python: 3.12.12
Pandas: 2.3.3
NumPy: 2.0.2


## 1. Configuration

Change these values if you want a smaller/larger demo.


In [2]:
PROJECT_NAME = "hm-recommender"

# Recommended demo size for a good frontend experience.
N_DEMO_CUSTOMERS = 100
N_RECOMMENDATIONS_PER_CUSTOMER = 30
N_SIMILAR_ITEMS_PER_ITEM = 20
N_SIMILAR_USERS_PER_USER = 10

# Max number of unique article cards to keep in the frontend demo dataset.
# The notebook may include more than recommendation articles because similar-items pages need extra articles.
MAX_DEMO_ARTICLES = 5000

# Image copy behavior.
COPY_DEMO_IMAGES = True
MAX_IMAGES_TO_COPY = 5000
IMAGE_EXTENSION = ".jpg"

# Export formats.
EXPORT_PARQUET = True
EXPORT_CSV = True
EXPORT_JSON = True

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("N_DEMO_CUSTOMERS:", N_DEMO_CUSTOMERS)
print("N_RECOMMENDATIONS_PER_CUSTOMER:", N_RECOMMENDATIONS_PER_CUSTOMER)
print("MAX_DEMO_ARTICLES:", MAX_DEMO_ARTICLES)
print("COPY_DEMO_IMAGES:", COPY_DEMO_IMAGES)


N_DEMO_CUSTOMERS: 100
N_RECOMMENDATIONS_PER_CUSTOMER: 30
MAX_DEMO_ARTICLES: 5000
COPY_DEMO_IMAGES: True


## 2. Environment and file search helpers

The notebook searches both `/kaggle/input` and `/kaggle/working`, so it can load artifacts from attached Kaggle datasets or previous notebook outputs.


In [3]:
def detect_environment() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None or Path("/kaggle/input").exists() or Path("/kaggle/working").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") is not None:
        return "colab"
    return "local"


ENVIRONMENT = detect_environment()
IN_KAGGLE = ENVIRONMENT == "kaggle"
IN_COLAB = ENVIRONMENT == "colab"

if IN_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME
elif IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
else:
    PROJECT_DIR = Path.cwd() / PROJECT_NAME

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
DEMO_EXPORT_DIR = PROJECT_DIR / "demo_export"
DEMO_DATA_DIR = DEMO_EXPORT_DIR / "data"
DEMO_PUBLIC_DIR = DEMO_EXPORT_DIR / "public"
DEMO_PUBLIC_IMAGES_DIR = DEMO_PUBLIC_DIR / "images"

for folder in [ARTIFACTS_DIR, DEMO_EXPORT_DIR, DEMO_DATA_DIR, DEMO_PUBLIC_IMAGES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [
    PROJECT_DIR,
    ARTIFACTS_DIR,
    Path.cwd(),
    Path("/kaggle/working"),
    Path("/kaggle/input"),
    Path("/content/drive/MyDrive"),
]

print("Detected environment:", ENVIRONMENT)
print("PROJECT_DIR:", PROJECT_DIR)
print("DEMO_EXPORT_DIR:", DEMO_EXPORT_DIR)
print("DEMO_DATA_DIR:", DEMO_DATA_DIR)
print("DEMO_PUBLIC_IMAGES_DIR:", DEMO_PUBLIC_IMAGES_DIR)


Detected environment: kaggle
PROJECT_DIR: /kaggle/working/hm-recommender
DEMO_EXPORT_DIR: /kaggle/working/hm-recommender/demo_export
DEMO_DATA_DIR: /kaggle/working/hm-recommender/demo_export/data
DEMO_PUBLIC_IMAGES_DIR: /kaggle/working/hm-recommender/demo_export/public/images


In [4]:
def find_file(filename: str, required: bool = False) -> Optional[Path]:
    candidate_subdirs = [
        "",
        "hm-recommender",
        "hm-recommender/data/processed",
        "hm-recommender/artifacts",
        "hm-recommender/artifacts/eda",
        "hm-recommender/artifacts/popularity_baseline",
        "hm-recommender/artifacts/als_recommender",
        "hm-recommender/artifacts/als_hyperparameter_tuning",
        "hm-recommender/artifacts/hybrid_als_popularity",
        "hm-recommender/artifacts/final_model_package",
        "hm-recommender/artifacts/similarity_modules",
    ]

    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for subdir in candidate_subdirs:
            candidate = root / subdir / filename
            if candidate.exists():
                return candidate

    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        try:
            matches = sorted(root.rglob(filename))
        except Exception:
            matches = []
        if matches:
            return matches[0]

    if required:
        raise FileNotFoundError(f"Could not find required file: {filename}")
    return None


def find_dir(dirname: str, required: bool = False) -> Optional[Path]:
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        direct = root / dirname
        if direct.exists() and direct.is_dir():
            return direct
        try:
            matches = sorted([p for p in root.rglob(dirname) if p.is_dir()])
        except Exception:
            matches = []
        if matches:
            return matches[0]
    if required:
        raise FileNotFoundError(f"Could not find required directory: {dirname}")
    return None


def find_processed_data_dir() -> Path:
    expected = find_file("train_interactions_aggregated.parquet", required=True)
    return expected.parent


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    print("Saved:", path)


def save_table(df: pd.DataFrame, stem: str) -> Dict[str, str]:
    saved = {}
    if EXPORT_PARQUET:
        path = DEMO_DATA_DIR / f"{stem}.parquet"
        df.to_parquet(path, index=False, compression="snappy")
        saved["parquet"] = str(path)
        print(f"Saved: {path} | {path.stat().st_size / (1024 ** 2):.4f} MB")
    if EXPORT_CSV:
        path = DEMO_DATA_DIR / f"{stem}.csv"
        df.to_csv(path, index=False)
        saved["csv"] = str(path)
        print(f"Saved: {path} | {path.stat().st_size / (1024 ** 2):.4f} MB")
    if EXPORT_JSON:
        path = DEMO_DATA_DIR / f"{stem}.json"
        df.to_json(path, orient="records", force_ascii=False)
        saved["json"] = str(path)
        print(f"Saved: {path} | {path.stat().st_size / (1024 ** 2):.4f} MB")
    return saved


def read_parquet_or_none(filename: str) -> Optional[pd.DataFrame]:
    path = find_file(filename)
    if path is None:
        print("Warning: missing", filename)
        return None
    print("Loaded", filename, "from", path)
    return pd.read_parquet(path)


def read_csv_or_none(filename: str) -> Optional[pd.DataFrame]:
    path = find_file(filename)
    if path is None:
        print("Warning: missing", filename)
        return None
    print("Loaded", filename, "from", path)
    return pd.read_csv(path)


def read_json_or_none(filename: str) -> Optional[dict]:
    path = find_file(filename)
    if path is None:
        print("Warning: missing", filename)
        return None
    print("Loaded", filename, "from", path)
    with open(path) as f:
        return json.load(f)


def memory_usage_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024 ** 2)


def print_df_info(df: pd.DataFrame, name: str, n: int = 5) -> None:
    print("\n" + name)
    print("-" * 100)
    print("Shape:", df.shape)
    print(f"Memory usage: {memory_usage_mb(df):.2f} MB")
    display(df.head(n))


## 3. Load required processed data and final artifacts


In [5]:
PROCESSED_DATA_DIR = find_processed_data_dir()
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)

articles = pd.read_parquet(PROCESSED_DATA_DIR / "articles_processed.parquet")
article_id_map = pd.read_parquet(PROCESSED_DATA_DIR / "article_id_map.parquet")
customer_id_map = pd.read_parquet(PROCESSED_DATA_DIR / "customer_id_map.parquet")

# Optional processed files.
train_interactions_path = PROCESSED_DATA_DIR / "train_interactions.parquet"
if train_interactions_path.exists():
    train_interactions = pd.read_parquet(train_interactions_path, columns=["customer_idx", "article_idx", "t_dat"])
    train_interactions["customer_idx"] = train_interactions["customer_idx"].astype("int32")
    train_interactions["article_idx"] = train_interactions["article_idx"].astype("int32")
    train_interactions["t_dat"] = pd.to_datetime(train_interactions["t_dat"])
else:
    train_interactions = pd.DataFrame()

# Final/hybrid outputs.
demo_hybrid_recommendations = read_parquet_or_none("demo_hybrid_recommendations.parquet")
hybrid_submission_like_demo = read_csv_or_none("hybrid_submission_like_demo.csv")
final_hybrid_leaderboard = read_csv_or_none("final_hybrid_leaderboard_k20.csv")
hybrid_tuning_leaderboard = read_csv_or_none("hybrid_tuning_leaderboard_k20.csv")
hybrid_decision_summary = read_json_or_none("hybrid_decision_summary.json")
best_final_model_info = read_json_or_none("best_final_model_info.json")

# Similarity outputs.
combined_similar_items = read_parquet_or_none("combined_similar_items_demo.parquet")
metadata_similar_items = read_parquet_or_none("metadata_similar_items_demo.parquet")
als_similar_items = read_parquet_or_none("als_similar_items_demo.parquet")
als_similar_users = read_parquet_or_none("als_similar_users_demo.parquet")
similar_user_taste_summary = read_parquet_or_none("similar_user_taste_summary.parquet")

# Optional EDA/model outputs.
eda_summary = read_json_or_none("eda_summary.json")
popularity_metrics = read_csv_or_none("popularity_baseline_metrics.csv")
als_metrics = read_csv_or_none("als_metrics.csv")

print_df_info(articles, "Articles")
print_df_info(customer_id_map, "Customer map")
if demo_hybrid_recommendations is not None:
    print_df_info(demo_hybrid_recommendations, "Demo hybrid recommendations")
if combined_similar_items is not None:
    print_df_info(combined_similar_items, "Combined similar items")
if als_similar_users is not None:
    print_df_info(als_similar_users, "ALS similar users")


PROCESSED_DATA_DIR: /kaggle/input/datasets/albaky7/hm-recommender/hm-recommender/data/processed
Loaded demo_hybrid_recommendations.parquet from /kaggle/input/datasets/albaky7/hm-final-model-package/hm-recommender/artifacts/final_model_package/demo_hybrid_recommendations.parquet
Loaded hybrid_submission_like_demo.csv from /kaggle/input/datasets/albaky7/hm-final-model-package/hm-recommender/artifacts/final_model_package/hybrid_submission_like_demo.csv
Loaded final_hybrid_leaderboard_k20.csv from /kaggle/input/datasets/albaky7/hm-final-model-package/hm-recommender/artifacts/final_model_package/final_hybrid_leaderboard_k20.csv
Loaded hybrid_tuning_leaderboard_k20.csv from /kaggle/input/datasets/albaky7/hm-final-model-package/hm-recommender/artifacts/final_model_package/hybrid_tuning_leaderboard_k20.csv
Loaded hybrid_decision_summary.json from /kaggle/input/datasets/albaky7/hm-final-model-package/hm-recommender/artifacts/final_model_package/hybrid_decision_summary.json
Loaded best_final_mod

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,article_idx
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,1
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,2
3,0110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",3
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",4



Customer map
----------------------------------------------------------------------------------------------------
Shape: (1371980, 2)
Memory usage: 153.09 MB


,customer_id,customer_idx
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,1
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,3
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,4



Demo hybrid recommendations
----------------------------------------------------------------------------------------------------
Shape: (20000, 17)
Memory usage: 13.42 MB


,customer_idx,article_idx,rank,model_name,base_als_model_name,als_weight,global_weight,recent90_weight,recency_weight,article_id,prod_name,product_type_name,product_group_name,colour_group_name,department_name,section_name,garment_group_name
0,0,15988,1,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0568597006,Hayes slim trouser,Trousers,Garment Lower body,Black,Trouser,Womens Tailoring,Trousers
1,0,42626,2,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0673677002,Henry polo. (1),Sweater,Garment Upper body,Black,Knitwear,Womens Tailoring,Knitwear
2,0,67522,3,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0751471001,Pluto RW slacks (1),Trousers,Garment Lower body,Black,Trouser,Womens Everyday Collection,Trousers
3,0,17383,4,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0573716012,Kanta slacks RW,Trousers,Garment Lower body,Black,Trouser,Womens Everyday Collection,Trousers
4,0,6641,5,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0507909001,Rebecca or Delphine shirt,Shirt,Garment Upper body,White,Blouse,Womens Tailoring,Blouses



Combined similar items
----------------------------------------------------------------------------------------------------
Shape: (10000, 13)
Memory usage: 5.67 MB


,query_article_idx,similar_article_idx,rank,similarity_score,method,query_article_id,similar_article_id,similar_prod_name,similar_product_type_name,similar_product_group_name,similar_colour_group_name,similar_section_name,similar_garment_group_name
0,0,1,1,0.967917,als_item_embedding_cosine,0108775015,0108775044,Strap top,Vest top,Garment Upper body,White,Womens Everyday Basics,Jersey Basic
1,0,10109,2,0.910952,als_item_embedding_cosine,0108775015,0538699001,V-neck strap top,Vest top,Garment Upper body,Black,Womens Everyday Basics,Jersey Basic
2,0,10111,3,0.877929,als_item_embedding_cosine,0108775015,0538699007,V-neck strap top,Vest top,Garment Upper body,White,Womens Everyday Basics,Jersey Basic
3,0,1512,4,0.864134,als_item_embedding_cosine,0108775015,0355072001,Anita Tank (1),Vest top,Garment Upper body,Black,Divided Basics,Jersey Basic
4,0,1513,5,0.843357,als_item_embedding_cosine,0108775015,0355072002,Anita Tank (1),Vest top,Garment Upper body,White,Divided Basics,Jersey Basic



ALS similar users
----------------------------------------------------------------------------------------------------
Shape: (1000, 5)
Memory usage: 0.10 MB


,query_customer_idx,similar_customer_idx,rank,similarity_score,method
0,1018839,977462,1,0.617626,als_user_embedding_cosine
1,1018839,527242,2,0.602845,als_user_embedding_cosine
2,1018839,865723,3,0.587437,als_user_embedding_cosine
3,1018839,15068,4,0.572817,als_user_embedding_cosine
4,1018839,593933,5,0.568770,als_user_embedding_cosine


## 4. Image path helpers

H&M image files usually follow this pattern:

```text
/images/{first_3_digits_of_article_id}/{article_id}.jpg
```

Example:

```text
article_id = 0706016001
frontend_image_url = /images/070/0706016001.jpg
```

The frontend uses this URL in an `<img>` tag. The actual copied file will live inside `demo_export/public/images/070/0706016001.jpg`.


In [6]:
def normalize_article_id(article_id) -> str:
    if pd.isna(article_id):
        return ""
    text = str(article_id).strip()
    if text.endswith(".0"):
        text = text[:-2]
    return text.zfill(10)


def article_id_to_image_subfolder(article_id: str) -> str:
    article_id = normalize_article_id(article_id)
    return article_id[:3]


def article_id_to_frontend_image_url(article_id: str) -> str:
    article_id = normalize_article_id(article_id)
    if not article_id:
        return "/images/placeholder-product.jpg"
    return f"/images/{article_id_to_image_subfolder(article_id)}/{article_id}{IMAGE_EXTENSION}"


def article_id_to_relative_image_path(article_id: str) -> Path:
    article_id = normalize_article_id(article_id)
    return Path(article_id_to_image_subfolder(article_id)) / f"{article_id}{IMAGE_EXTENSION}"


def find_hm_images_dir() -> Optional[Path]:
    # Common Kaggle location: /kaggle/input/h-and-m-personalized-fashion-recommendations/images
    candidate_names = ["images"]
    for name in candidate_names:
        found = find_dir(name, required=False)
        if found is not None:
            # A valid H&M images dir should contain many numeric subfolders.
            numeric_subdirs = [p for p in found.iterdir() if p.is_dir() and p.name.isdigit()]
            if len(numeric_subdirs) > 10:
                return found
    return None


HM_IMAGES_DIR = find_hm_images_dir()
print("HM_IMAGES_DIR:", HM_IMAGES_DIR)

# Add image URL to articles.
articles = articles.copy()
articles["article_id"] = articles["article_id"].apply(normalize_article_id)
articles["image_url"] = articles["article_id"].apply(article_id_to_frontend_image_url)
articles["image_relative_path"] = articles["article_id"].apply(lambda x: str(article_id_to_relative_image_path(x)))

print(articles[["article_idx", "article_id", "image_url", "image_relative_path"]].head())


HM_IMAGES_DIR: None
   article_idx  article_id                   image_url image_relative_path
0            0  0108775015  /images/010/0108775015.jpg  010/0108775015.jpg
1            1  0108775044  /images/010/0108775044.jpg  010/0108775044.jpg
2            2  0108775051  /images/010/0108775051.jpg  010/0108775051.jpg
3            3  0110065001  /images/011/0110065001.jpg  011/0110065001.jpg
4            4  0110065002  /images/011/0110065002.jpg  011/0110065002.jpg


## 5. Select demo customers

We select customers that have hybrid recommendations. If similar-user output exists, we prioritize customers that also have similar-user data.


In [7]:
if demo_hybrid_recommendations is None or len(demo_hybrid_recommendations) == 0:
    raise FileNotFoundError("demo_hybrid_recommendations.parquet is required for a strong frontend demo.")

# Normalize columns.
demo_hybrid_recommendations = demo_hybrid_recommendations.copy()
demo_hybrid_recommendations["customer_idx"] = demo_hybrid_recommendations["customer_idx"].astype("int32")
demo_hybrid_recommendations["article_idx"] = demo_hybrid_recommendations["article_idx"].astype("int32")
demo_hybrid_recommendations["rank"] = demo_hybrid_recommendations["rank"].astype("int32")

candidate_customers = demo_hybrid_recommendations["customer_idx"].drop_duplicates().astype("int32").tolist()

if als_similar_users is not None and len(als_similar_users) > 0:
    customers_with_similar_users = set(als_similar_users["query_customer_idx"].astype("int32"))
    prioritized = [c for c in candidate_customers if c in customers_with_similar_users]
    remaining = [c for c in candidate_customers if c not in customers_with_similar_users]
    selected_customer_indices = (prioritized + remaining)[:N_DEMO_CUSTOMERS]
else:
    selected_customer_indices = candidate_customers[:N_DEMO_CUSTOMERS]

selected_customer_indices = [int(c) for c in selected_customer_indices]
print("Selected demo customers:", len(selected_customer_indices))
print(selected_customer_indices[:20])

# Build customer table.
demo_customers = customer_id_map[customer_id_map["customer_idx"].isin(selected_customer_indices)].copy()
demo_customers["customer_idx"] = demo_customers["customer_idx"].astype("int32")

# Add lightweight customer labels for UI.
demo_customers["display_name"] = demo_customers["customer_idx"].apply(lambda x: f"Demo Customer {int(x)}")
demo_customers["customer_segment"] = "Fashion Shopper"

# Add basic customer activity info if train interactions are available.
if len(train_interactions) > 0:
    user_activity = (
        train_interactions[train_interactions["customer_idx"].isin(selected_customer_indices)]
        .groupby("customer_idx", as_index=False)
        .agg(
            train_purchase_events=("article_idx", "size"),
            train_unique_items=("article_idx", "nunique"),
            last_train_purchase_date=("t_dat", "max"),
        )
    )
    demo_customers = demo_customers.merge(user_activity, on="customer_idx", how="left")

print_df_info(demo_customers, "Demo customers")
save_table(demo_customers, "demo_customers")


Selected demo customers: 100
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

Demo customers
----------------------------------------------------------------------------------------------------
Shape: (100, 7)
Memory usage: 0.03 MB


,customer_id,customer_idx,display_name,customer_segment,train_purchase_events,train_unique_items,last_train_purchase_date
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,Demo Customer 0,Fashion Shopper,15,13,2019-11-28
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,1,Demo Customer 1,Fashion Shopper,70,56,2020-01-25
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2,Demo Customer 2,Fashion Shopper,7,5,2020-02-03
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,3,Demo Customer 3,Fashion Shopper,2,2,2019-06-09
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,4,Demo Customer 4,Fashion Shopper,6,5,2019-10-09


Saved: /kaggle/working/hm-recommender/demo_export/data/demo_customers.parquet | 0.0133 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_customers.csv | 0.0113 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_customers.json | 0.0251 MB


{'parquet': '/kaggle/working/hm-recommender/demo_export/data/demo_customers.parquet',
 'csv': '/kaggle/working/hm-recommender/demo_export/data/demo_customers.csv',
 'json': '/kaggle/working/hm-recommender/demo_export/data/demo_customers.json'}

## 6. Build home-feed recommendations for selected demo customers


In [8]:
# Keep top recommendations per selected customer.
demo_home_feed = demo_hybrid_recommendations[
    demo_hybrid_recommendations["customer_idx"].isin(selected_customer_indices)
].copy()

demo_home_feed = demo_home_feed.sort_values(["customer_idx", "rank"]).groupby("customer_idx").head(N_RECOMMENDATIONS_PER_CUSTOMER)

# Ensure article_id and metadata are attached consistently.
article_base_cols = [
    "article_idx",
    "article_id",
    "prod_name",
    "product_type_name",
    "product_group_name",
    "graphical_appearance_name",
    "colour_group_name",
    "department_name",
    "index_name",
    "section_name",
    "garment_group_name",
    "detail_desc",
    "image_url",
    "image_relative_path",
]
article_base_cols = [col for col in article_base_cols if col in articles.columns]
article_base = articles[article_base_cols].copy()

# Drop duplicated metadata columns if already present from previous notebooks.
metadata_cols_to_drop = [col for col in article_base_cols if col in demo_home_feed.columns and col not in ["article_idx"]]
demo_home_feed = demo_home_feed.drop(columns=metadata_cols_to_drop, errors="ignore")

demo_home_feed = demo_home_feed.merge(article_base, on="article_idx", how="left", validate="many_to_one")

demo_home_feed["recommendation_type"] = "personalized_hybrid"
demo_home_feed["card_title"] = demo_home_feed.get("prod_name", pd.Series("Product", index=demo_home_feed.index)).fillna("Product")

print_df_info(demo_home_feed, "Demo home feed")
save_table(demo_home_feed, "demo_home_feed")



Demo home feed
----------------------------------------------------------------------------------------------------
Shape: (2000, 24)
Memory usage: 2.39 MB


,customer_idx,article_idx,rank,model_name,base_als_model_name,als_weight,global_weight,recent90_weight,recency_weight,article_id,prod_name,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,department_name,index_name,section_name,garment_group_name,detail_desc,image_url,image_relative_path,recommendation_type,card_title
0,0,15988,1,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0568597006,Hayes slim trouser,Trousers,Garment Lower body,Solid,Black,Trouser,Ladieswear,Womens Tailoring,Trousers,Suit trousers in a stretch weave with a regula...,/images/056/0568597006.jpg,056/0568597006.jpg,personalized_hybrid,Hayes slim trouser
1,0,42626,2,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0673677002,Henry polo. (1),Sweater,Garment Upper body,Solid,Black,Knitwear,Ladieswear,Womens Tailoring,Knitwear,"Jumper in a soft, fine knit with a ribbed polo...",/images/067/0673677002.jpg,067/0673677002.jpg,personalized_hybrid,Henry polo. (1)
2,0,67522,3,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0751471001,Pluto RW slacks (1),Trousers,Garment Lower body,Solid,Black,Trouser,Ladieswear,Womens Everyday Collection,Trousers,Ankle-length cigarette trousers in a stretch w...,/images/075/0751471001.jpg,075/0751471001.jpg,personalized_hybrid,Pluto RW slacks (1)
3,0,17383,4,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0573716012,Kanta slacks RW,Trousers,Garment Lower body,Solid,Black,Trouser,Ladieswear,Womens Everyday Collection,Trousers,Ankle-length cigarette trousers in a stretch w...,/images/057/0573716012.jpg,057/0573716012.jpg,personalized_hybrid,Kanta slacks RW
4,0,6641,5,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0507909001,Rebecca or Delphine shirt,Shirt,Garment Upper body,Solid,White,Blouse,Ladieswear,Womens Tailoring,Blouses,Gently tailored shirt in a stretch cotton blen...,/images/050/0507909001.jpg,050/0507909001.jpg,personalized_hybrid,Rebecca or Delphine shirt


Saved: /kaggle/working/hm-recommender/demo_export/data/demo_home_feed.parquet | 0.0998 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_home_feed.csv | 0.7965 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_home_feed.json | 1.6304 MB


{'parquet': '/kaggle/working/hm-recommender/demo_export/data/demo_home_feed.parquet',
 'csv': '/kaggle/working/hm-recommender/demo_export/data/demo_home_feed.csv',
 'json': '/kaggle/working/hm-recommender/demo_export/data/demo_home_feed.json'}

## 7. Select demo article catalog

The frontend should not load all 100k+ items. We export a curated catalog containing:

- recommended feed items,
- similar-item query items,
- similar-item result items,
- and optionally a few extra popular items.


In [9]:
article_indices_needed = set(demo_home_feed["article_idx"].dropna().astype("int32"))

if combined_similar_items is not None and len(combined_similar_items) > 0:
    for col in ["query_article_idx", "similar_article_idx"]:
        if col in combined_similar_items.columns:
            article_indices_needed.update(combined_similar_items[col].dropna().astype("int32"))

# Add some popular train items if we still have room.
if len(train_interactions) > 0 and len(article_indices_needed) < MAX_DEMO_ARTICLES:
    extra_popular = train_interactions["article_idx"].value_counts().index.astype("int32").tolist()
    for article_idx in extra_popular:
        article_indices_needed.add(int(article_idx))
        if len(article_indices_needed) >= MAX_DEMO_ARTICLES:
            break

# Limit to max count while preserving home-feed articles first.
home_article_order = demo_home_feed.sort_values(["customer_idx", "rank"])["article_idx"].drop_duplicates().astype("int32").tolist()
other_articles = [a for a in article_indices_needed if a not in set(home_article_order)]
selected_article_indices = (home_article_order + other_articles)[:MAX_DEMO_ARTICLES]
selected_article_indices = [int(a) for a in selected_article_indices]

print("Selected demo articles:", len(selected_article_indices))

demo_articles = articles[articles["article_idx"].isin(selected_article_indices)].copy()
demo_articles["article_idx"] = demo_articles["article_idx"].astype("int32")

# Add flags useful for frontend filtering.
demo_articles["in_home_feed"] = demo_articles["article_idx"].isin(set(demo_home_feed["article_idx"].astype("int32")))
if combined_similar_items is not None and len(combined_similar_items) > 0:
    demo_articles["in_similar_items"] = demo_articles["article_idx"].isin(
        set(combined_similar_items.get("query_article_idx", pd.Series(dtype="int32")).dropna().astype("int32"))
        | set(combined_similar_items.get("similar_article_idx", pd.Series(dtype="int32")).dropna().astype("int32"))
    )
else:
    demo_articles["in_similar_items"] = False

print_df_info(demo_articles, "Demo articles")
save_table(demo_articles, "demo_articles")


Selected demo articles: 5000

Demo articles
----------------------------------------------------------------------------------------------------
Shape: (5000, 30)
Memory usage: 5.87 MB


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,article_idx,image_url,image_relative_path,in_home_feed,in_similar_items
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0,/images/010/0108775015.jpg,010/0108775015.jpg,True,True
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,1,/images/010/0108775044.jpg,010/0108775044.jpg,False,True
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,2,/images/010/0108775051.jpg,010/0108775051.jpg,False,True
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",4,/images/011/0110065002.jpg,011/0110065002.jpg,False,True
6,0111565001,111565,20 den 1p Stockings,304,Underwear Tights,Socks & Tights,1010016,Solid,9,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,"Semi shiny nylon stockings with a wide, reinfo...",6,/images/011/0111565001.jpg,011/0111565001.jpg,False,True


Saved: /kaggle/working/hm-recommender/demo_export/data/demo_articles.parquet | 0.3262 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_articles.csv | 1.7833 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_articles.json | 4.6886 MB


{'parquet': '/kaggle/working/hm-recommender/demo_export/data/demo_articles.parquet',
 'csv': '/kaggle/working/hm-recommender/demo_export/data/demo_articles.csv',
 'json': '/kaggle/working/hm-recommender/demo_export/data/demo_articles.json'}

## 8. Export similar items for demo articles


In [10]:
if combined_similar_items is not None and len(combined_similar_items) > 0:
    demo_similar_items = combined_similar_items[
        combined_similar_items["query_article_idx"].isin(selected_article_indices)
        & combined_similar_items["similar_article_idx"].isin(selected_article_indices)
    ].copy()

    # Keep top N per query/method.
    demo_similar_items = (
        demo_similar_items.sort_values(["query_article_idx", "method", "rank"])
        .groupby(["query_article_idx", "method"])
        .head(N_SIMILAR_ITEMS_PER_ITEM)
        .reset_index(drop=True)
    )

    # Attach image URLs for query and similar items.
    query_meta = articles[["article_idx", "article_id", "prod_name", "image_url"]].rename(
        columns={
            "article_idx": "query_article_idx",
            "article_id": "query_article_id",
            "prod_name": "query_prod_name",
            "image_url": "query_image_url",
        }
    )
    similar_meta = articles[["article_idx", "article_id", "prod_name", "image_url"]].rename(
        columns={
            "article_idx": "similar_article_idx",
            "article_id": "similar_article_id",
            "prod_name": "similar_prod_name",
            "image_url": "similar_image_url",
        }
    )

    for col in ["query_article_id", "query_prod_name", "query_image_url", "similar_article_id", "similar_prod_name", "similar_image_url"]:
        demo_similar_items = demo_similar_items.drop(columns=[col], errors="ignore")

    demo_similar_items = demo_similar_items.merge(query_meta, on="query_article_idx", how="left")
    demo_similar_items = demo_similar_items.merge(similar_meta, on="similar_article_idx", how="left")
else:
    demo_similar_items = pd.DataFrame()

print_df_info(demo_similar_items, "Demo similar items")
save_table(demo_similar_items, "demo_similar_items")



Demo similar items
----------------------------------------------------------------------------------------------------
Shape: (10000, 16)
Memory usage: 7.74 MB


,query_article_idx,similar_article_idx,rank,similarity_score,method,similar_product_type_name,similar_product_group_name,similar_colour_group_name,similar_section_name,similar_garment_group_name,query_article_id,query_prod_name,query_image_url,similar_article_id,similar_prod_name,similar_image_url
0,0,1,1,0.967917,als_item_embedding_cosine,Vest top,Garment Upper body,White,Womens Everyday Basics,Jersey Basic,0108775015,Strap top,/images/010/0108775015.jpg,0108775044,Strap top,/images/010/0108775044.jpg
1,0,10109,2,0.910952,als_item_embedding_cosine,Vest top,Garment Upper body,Black,Womens Everyday Basics,Jersey Basic,0108775015,Strap top,/images/010/0108775015.jpg,0538699001,V-neck strap top,/images/053/0538699001.jpg
2,0,10111,3,0.877929,als_item_embedding_cosine,Vest top,Garment Upper body,White,Womens Everyday Basics,Jersey Basic,0108775015,Strap top,/images/010/0108775015.jpg,0538699007,V-neck strap top,/images/053/0538699007.jpg
3,0,1512,4,0.864134,als_item_embedding_cosine,Vest top,Garment Upper body,Black,Divided Basics,Jersey Basic,0108775015,Strap top,/images/010/0108775015.jpg,0355072001,Anita Tank (1),/images/035/0355072001.jpg
4,0,1513,5,0.843357,als_item_embedding_cosine,Vest top,Garment Upper body,White,Divided Basics,Jersey Basic,0108775015,Strap top,/images/010/0108775015.jpg,0355072002,Anita Tank (1),/images/035/0355072002.jpg


Saved: /kaggle/working/hm-recommender/demo_export/data/demo_similar_items.parquet | 0.2589 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_similar_items.csv | 2.2574 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_similar_items.json | 5.6471 MB


{'parquet': '/kaggle/working/hm-recommender/demo_export/data/demo_similar_items.parquet',
 'csv': '/kaggle/working/hm-recommender/demo_export/data/demo_similar_items.csv',
 'json': '/kaggle/working/hm-recommender/demo_export/data/demo_similar_items.json'}

## 9. Export similar users and taste summaries


In [21]:
# ============================================================
# FIX: Export similar users even when home-feed demo customers
# do not overlap with the 08 similarity-module customers
# ============================================================

import pandas as pd

N_USER_PAGE_CUSTOMERS = 50
N_SIMILAR_USERS_PER_USER = globals().get("N_SIMILAR_USERS_PER_USER", 10)

if als_similar_users is None or len(als_similar_users) == 0:
    raise ValueError("als_similar_users is empty or not loaded. Check 08-similarity-modules output.")

# Keep the original home-feed customers
home_feed_customer_indices = list(selected_customer_indices)

# Customers that actually have similar-user results from 08
available_similar_user_customers = (
    als_similar_users["query_customer_idx"]
    .dropna()
    .astype("int32")
    .drop_duplicates()
    .tolist()
)

print("Home-feed demo customers:", len(home_feed_customer_indices))
print("Customers available in similar-user file:", len(available_similar_user_customers))
print("Sample similar-user query customers:", available_similar_user_customers[:20])

# First try overlap
overlap_customers = [
    c for c in home_feed_customer_indices
    if c in set(available_similar_user_customers)
]

print("Overlap customers:", len(overlap_customers))

# If no overlap, use customers from the similar-user file for the User Page
if len(overlap_customers) > 0:
    user_page_customer_indices = overlap_customers[:N_USER_PAGE_CUSTOMERS]
else:
    user_page_customer_indices = available_similar_user_customers[:N_USER_PAGE_CUSTOMERS]

print("Selected user-page customers:", len(user_page_customer_indices))
print("User-page customers sample:", user_page_customer_indices[:20])

# Update selected_customer_indices so demo_customers contains both:
# 1. home feed customers
# 2. user page customers
selected_customer_indices = sorted(
    set(home_feed_customer_indices) | set(user_page_customer_indices)
)

print("Total demo customers after union:", len(selected_customer_indices))

# ------------------------------------------------------------
# Rebuild and resave demo_customers with both groups
# ------------------------------------------------------------

demo_customers = customer_id_map[
    customer_id_map["customer_idx"].isin(selected_customer_indices)
].copy()

demo_customers["customer_idx"] = demo_customers["customer_idx"].astype("int32")
demo_customers["display_name"] = demo_customers["customer_idx"].apply(
    lambda x: f"Demo Customer {int(x)}"
)

def assign_demo_customer_segment(customer_idx):
    if customer_idx in set(home_feed_customer_indices) and customer_idx in set(user_page_customer_indices):
        return "Home Feed + Similar Users"
    if customer_idx in set(home_feed_customer_indices):
        return "Home Feed Customer"
    if customer_idx in set(user_page_customer_indices):
        return "Similar User Demo Customer"
    return "Fashion Shopper"

demo_customers["customer_segment"] = demo_customers["customer_idx"].apply(assign_demo_customer_segment)

# Add train activity info again
if "train_interactions" in globals() and len(train_interactions) > 0:
    user_activity = (
        train_interactions[train_interactions["customer_idx"].isin(selected_customer_indices)]
        .groupby("customer_idx", as_index=False)
        .agg(
            train_purchase_events=("article_idx", "size"),
            train_unique_items=("article_idx", "nunique"),
            last_train_purchase_date=("t_dat", "max"),
        )
    )
    demo_customers = demo_customers.merge(user_activity, on="customer_idx", how="left")

print_df_info(demo_customers, "Updated demo customers")
save_table(demo_customers, "demo_customers")

# ------------------------------------------------------------
# Rebuild and resave demo_similar_users
# ------------------------------------------------------------

demo_similar_users = als_similar_users[
    als_similar_users["query_customer_idx"].isin(user_page_customer_indices)
].copy()

demo_similar_users = (
    demo_similar_users.sort_values(["query_customer_idx", "rank"])
    .groupby("query_customer_idx")
    .head(N_SIMILAR_USERS_PER_USER)
    .reset_index(drop=True)
)

print_df_info(demo_similar_users, "Fixed demo similar users")
save_table(demo_similar_users, "demo_similar_users")

# ------------------------------------------------------------
# Rebuild and resave similar-user taste summary
# ------------------------------------------------------------

if similar_user_taste_summary is not None and len(similar_user_taste_summary) > 0:
    demo_similar_user_taste_summary = similar_user_taste_summary[
        similar_user_taste_summary["query_customer_idx"].isin(user_page_customer_indices)
    ].copy()

    demo_similar_user_taste_summary = (
        demo_similar_user_taste_summary.sort_values(["query_customer_idx", "rank"])
        .groupby("query_customer_idx")
        .head(N_SIMILAR_USERS_PER_USER)
        .reset_index(drop=True)
    )
else:
    demo_similar_user_taste_summary = pd.DataFrame()

print_df_info(demo_similar_user_taste_summary, "Fixed demo similar user taste summary")
save_table(demo_similar_user_taste_summary, "demo_similar_user_taste_summary")

print("\nFix completed.")
print("demo_similar_users rows:", len(demo_similar_users))
print("demo_similar_user_taste_summary rows:", len(demo_similar_user_taste_summary))

Home-feed demo customers: 100
Customers available in similar-user file: 50
Sample similar-user query customers: [1018839, 281710, 969180, 950285, 1035425, 394603, 63928, 711274, 760470, 459261, 1218201, 261666, 736963, 1234000, 891429, 582904, 545477, 1098652, 1284194, 1114704]
Overlap customers: 0
Selected user-page customers: 50
User-page customers sample: [1018839, 281710, 969180, 950285, 1035425, 394603, 63928, 711274, 760470, 459261, 1218201, 261666, 736963, 1234000, 891429, 582904, 545477, 1098652, 1284194, 1114704]
Total demo customers after union: 150

Updated demo customers
----------------------------------------------------------------------------------------------------
Shape: (150, 7)
Memory usage: 0.04 MB


,customer_id,customer_idx,display_name,customer_segment,train_purchase_events,train_unique_items,last_train_purchase_date
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,Demo Customer 0,Home Feed Customer,15,13,2019-11-28
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,1,Demo Customer 1,Home Feed Customer,70,56,2020-01-25
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2,Demo Customer 2,Home Feed Customer,7,5,2020-02-03
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,3,Demo Customer 3,Home Feed Customer,2,2,2019-06-09
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,4,Demo Customer 4,Home Feed Customer,6,5,2019-10-09


Saved: /kaggle/working/hm-recommender/demo_export/data/demo_customers.parquet | 0.0178 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_customers.csv | 0.0182 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_customers.json | 0.0390 MB

Fixed demo similar users
----------------------------------------------------------------------------------------------------
Shape: (500, 5)
Memory usage: 0.05 MB


,query_customer_idx,similar_customer_idx,rank,similarity_score,method
0,20300,1268154,1,0.682627,als_user_embedding_cosine
1,20300,1098634,2,0.667954,als_user_embedding_cosine
2,20300,1116034,3,0.660589,als_user_embedding_cosine
3,20300,807928,4,0.658946,als_user_embedding_cosine
4,20300,1364275,5,0.658946,als_user_embedding_cosine


Saved: /kaggle/working/hm-recommender/demo_export/data/demo_similar_users.parquet | 0.0100 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_similar_users.csv | 0.0293 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_similar_users.json | 0.0660 MB

Fixed demo similar user taste summary
----------------------------------------------------------------------------------------------------
Shape: (250, 10)
Memory usage: 0.16 MB


,query_customer_idx,similar_customer_idx,rank,similarity_score,query_top_garment_groups,similar_top_garment_groups,common_garment_groups,query_top_sections,similar_top_sections,common_sections
0,20300,1268154,1,0.682627,Blouses | Jersey Fancy | Trousers | Dresses La...,Unknown,,Womens Everyday Collection | Womens Trend | Di...,Womens Everyday Collection,Womens Everyday Collection
1,20300,1098634,2,0.667954,Blouses | Jersey Fancy | Trousers | Dresses La...,Trousers,Trousers,Womens Everyday Collection | Womens Trend | Di...,Womens Trend,Womens Trend
2,20300,1116034,3,0.660589,Blouses | Jersey Fancy | Trousers | Dresses La...,Jersey Basic | Trousers | Blouses | Outdoor | ...,Blouses | Trousers | Jersey Basic,Womens Everyday Collection | Womens Trend | Di...,Womens Everyday Basics | Womens Trend | Divide...,Womens Trend | Divided Collection | Womens Jac...
3,20300,807928,4,0.658946,Blouses | Jersey Fancy | Trousers | Dresses La...,Trousers,Trousers,Womens Everyday Collection | Womens Trend | Di...,Womens Trend,Womens Trend
4,20300,1364275,5,0.658946,Blouses | Jersey Fancy | Trousers | Dresses La...,Trousers,Trousers,Womens Everyday Collection | Womens Trend | Di...,Womens Trend,Womens Trend


Saved: /kaggle/working/hm-recommender/demo_export/data/demo_similar_user_taste_summary.parquet | 0.0205 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_similar_user_taste_summary.csv | 0.0954 MB
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_similar_user_taste_summary.json | 0.1472 MB

Fix completed.
demo_similar_users rows: 500
demo_similar_user_taste_summary rows: 250


In [ ]:
# if als_similar_users is not None and len(als_similar_users) > 0:
#     demo_similar_users = als_similar_users[
#         als_similar_users["query_customer_idx"].isin(selected_customer_indices)
#     ].copy()
#     demo_similar_users = (
#         demo_similar_users.sort_values(["query_customer_idx", "rank"])
#         .groupby("query_customer_idx")
#         .head(N_SIMILAR_USERS_PER_USER)
#         .reset_index(drop=True)
#     )
# else:
#     demo_similar_users = pd.DataFrame()

# if similar_user_taste_summary is not None and len(similar_user_taste_summary) > 0:
#     demo_similar_user_taste_summary = similar_user_taste_summary[
#         similar_user_taste_summary["query_customer_idx"].isin(selected_customer_indices)
#     ].copy()
#     demo_similar_user_taste_summary = (
#         demo_similar_user_taste_summary.sort_values(["query_customer_idx", "rank"])
#         .groupby("query_customer_idx")
#         .head(N_SIMILAR_USERS_PER_USER)
#         .reset_index(drop=True)
#     )
# else:
#     demo_similar_user_taste_summary = pd.DataFrame()

# print_df_info(demo_similar_users, "Demo similar users")
# print_df_info(demo_similar_user_taste_summary, "Demo similar user taste summary")
# save_table(demo_similar_users, "demo_similar_users")
# save_table(demo_similar_user_taste_summary, "demo_similar_user_taste_summary")


## 10. Export metrics and summary JSON files


In [12]:
demo_model_metrics = {
    "final_model_name": None,
    "final_model_family": "hybrid",
    "best_final_model_info": best_final_model_info,
    "hybrid_decision_summary": hybrid_decision_summary,
    "primary_metrics": {},
}

if final_hybrid_leaderboard is not None and len(final_hybrid_leaderboard) > 0:
    best_hybrid_row = final_hybrid_leaderboard.iloc[0].to_dict()
    demo_model_metrics["final_model_name"] = best_hybrid_row.get("hybrid_name", best_hybrid_row.get("model_name", "hybrid_als40_global40_recent20"))
    demo_model_metrics["primary_metrics"] = best_hybrid_row
elif best_final_model_info is not None:
    demo_model_metrics["final_model_name"] = best_final_model_info.get("final_model_name")

# Add comparison tables as records for easy frontend rendering.
if final_hybrid_leaderboard is not None:
    demo_model_metrics["final_hybrid_leaderboard"] = final_hybrid_leaderboard.head(20).to_dict(orient="records")
if hybrid_tuning_leaderboard is not None:
    demo_model_metrics["hybrid_tuning_leaderboard"] = hybrid_tuning_leaderboard.head(20).to_dict(orient="records")
if popularity_metrics is not None:
    demo_model_metrics["popularity_baseline_sample"] = popularity_metrics.head(20).to_dict(orient="records")
if als_metrics is not None:
    demo_model_metrics["als_metrics_sample"] = als_metrics.head(20).to_dict(orient="records")

save_json(demo_model_metrics, DEMO_DATA_DIR / "demo_model_metrics.json")

# EDA summary fallback if original EDA summary is missing.
if eda_summary is None:
    eda_summary = {
        "articles_total": int(len(articles)),
        "demo_articles_total": int(len(demo_articles)),
        "demo_customers_total": int(len(demo_customers)),
        "demo_home_feed_rows": int(len(demo_home_feed)),
        "demo_similar_items_rows": int(len(demo_similar_items)),
        "demo_similar_users_rows": int(len(demo_similar_users)),
    }
else:
    eda_summary = dict(eda_summary)
    eda_summary.update({
        "demo_articles_total": int(len(demo_articles)),
        "demo_customers_total": int(len(demo_customers)),
        "demo_home_feed_rows": int(len(demo_home_feed)),
        "demo_similar_items_rows": int(len(demo_similar_items)),
        "demo_similar_users_rows": int(len(demo_similar_users)),
    })

save_json(eda_summary, DEMO_DATA_DIR / "demo_eda_summary.json")


Saved: /kaggle/working/hm-recommender/demo_export/data/demo_model_metrics.json
Saved: /kaggle/working/hm-recommender/demo_export/data/demo_eda_summary.json


## 11. Copy only needed demo images

This is the important part for the frontend.

Instead of downloading/copying the full 28GB image folder, this cell copies only the images for `demo_articles` into:

```text
/kaggle/working/hm-recommender/demo_export/public/images/
```

The final React app can place this folder under:

```text
frontend/public/images/
```

Then product cards can use `image_url`, for example:

```text
/images/070/0706016001.jpg
```


In [18]:
# ============================================================
# Copy only the needed demo images for the React frontend
# ============================================================

from pathlib import Path
import shutil
import pandas as pd

# Correct Kaggle path from your diagnostic output
HM_IMAGES_DIR = Path("/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations/images")

# Output folder for the web app images
DEMO_EXPORT_DIR = Path("/kaggle/working/hm-recommender/demo_export")
DEMO_PUBLIC_DIR = DEMO_EXPORT_DIR / "public"
DEMO_PUBLIC_IMAGES_DIR = DEMO_PUBLIC_DIR / "images"

DEMO_PUBLIC_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print("HM_IMAGES_DIR:", HM_IMAGES_DIR)
print("Exists:", HM_IMAGES_DIR.exists())

if HM_IMAGES_DIR.exists():
    print("Sample source image folders:", sorted([p.name for p in HM_IMAGES_DIR.iterdir() if p.is_dir()])[:10])
else:
    raise FileNotFoundError(f"H&M images directory not found: {HM_IMAGES_DIR}")


def normalize_article_id(article_id) -> str:
    """
    H&M article_id should be a 10-digit string.
    Sometimes pandas reads it as int, so we convert and pad with zeros.
    """
    if pd.isna(article_id):
        return ""
    article_id = str(article_id).strip()
    article_id = article_id.split(".")[0]  # handles accidental float-like values
    return article_id.zfill(10)


def article_id_to_relative_image_path(article_id) -> str:
    """
    Example:
    article_id = 0706016001
    relative path = images/070/0706016001.jpg
    """
    article_id = normalize_article_id(article_id)
    folder = article_id[:3]
    return f"images/{folder}/{article_id}.jpg"


def article_id_to_frontend_url(article_id) -> str:
    """
    React frontend will use this URL.
    Example:
    /images/070/0706016001.jpg
    """
    article_id = normalize_article_id(article_id)
    folder = article_id[:3]
    return f"/images/{folder}/{article_id}.jpg"


def source_image_path_for_article_id(article_id) -> Path:
    """
    Source path inside Kaggle H&M dataset.
    Example:
    /kaggle/input/.../images/070/0706016001.jpg
    """
    article_id = normalize_article_id(article_id)
    folder = article_id[:3]
    return HM_IMAGES_DIR / folder / f"{article_id}.jpg"


def destination_image_path_for_article_id(article_id) -> Path:
    """
    Destination path inside demo export folder.
    Example:
    /kaggle/working/hm-recommender/demo_export/public/images/070/0706016001.jpg
    """
    article_id = normalize_article_id(article_id)
    folder = article_id[:3]
    return DEMO_PUBLIC_IMAGES_DIR / folder / f"{article_id}.jpg"


# ------------------------------------------------------------
# Collect article_ids that the website will actually show
# ------------------------------------------------------------

article_ids_to_copy = set()

# Main product catalog used by frontend
if "demo_articles" in globals() and "article_id" in demo_articles.columns:
    article_ids_to_copy.update(demo_articles["article_id"].dropna().map(normalize_article_id).tolist())

# Home feed recommendation cards
if "demo_home_feed" in globals() and "article_id" in demo_home_feed.columns:
    article_ids_to_copy.update(demo_home_feed["article_id"].dropna().map(normalize_article_id).tolist())

# Similar item cards
if "demo_similar_items" in globals():
    if "similar_article_id" in demo_similar_items.columns:
        article_ids_to_copy.update(demo_similar_items["similar_article_id"].dropna().map(normalize_article_id).tolist())
    if "query_article_id" in demo_similar_items.columns:
        article_ids_to_copy.update(demo_similar_items["query_article_id"].dropna().map(normalize_article_id).tolist())
    if "article_id" in demo_similar_items.columns:
        article_ids_to_copy.update(demo_similar_items["article_id"].dropna().map(normalize_article_id).tolist())

# Remove bad/empty IDs
article_ids_to_copy = sorted([x for x in article_ids_to_copy if x and x != "0000000000"])

print("Images selected for copy:", len(article_ids_to_copy))
print("Sample article_ids:", article_ids_to_copy[:10])


# ------------------------------------------------------------
# Copy only selected images
# ------------------------------------------------------------

copy_rows = []
copied_count = 0
already_exists_count = 0
missing_count = 0
skipped_existing_count = 0

for i, article_id in enumerate(article_ids_to_copy, start=1):
    source_path = source_image_path_for_article_id(article_id)
    destination_path = destination_image_path_for_article_id(article_id)

    destination_path.parent.mkdir(parents=True, exist_ok=True)

    if not source_path.exists():
        missing_count += 1
        status = "missing_source"
    else:
        if destination_path.exists():
            already_exists_count += 1
            skipped_existing_count += 1
            status = "already_exists"
        else:
            shutil.copy2(source_path, destination_path)
            copied_count += 1
            status = "copied"

    copy_rows.append(
        {
            "article_id": article_id,
            "source_path": str(source_path),
            "frontend_url": article_id_to_frontend_url(article_id),
            "destination_path": str(destination_path),
            "status": status,
        }
    )

    if i % 500 == 0:
        print(f"Processed {i}/{len(article_ids_to_copy)} images...")


image_copy_report = pd.DataFrame(copy_rows)

print("\nImage copy finished.")
print("Copied:", copied_count)
print("Already existed:", already_exists_count)
print("Missing:", missing_count)
print("Total processed:", len(image_copy_report))

# Save report
DEMO_DATA_DIR = DEMO_EXPORT_DIR / "data"
DEMO_DATA_DIR.mkdir(parents=True, exist_ok=True)

image_copy_report.to_parquet(DEMO_DATA_DIR / "image_copy_report.parquet", index=False)
image_copy_report.to_csv(DEMO_DATA_DIR / "image_copy_report.csv", index=False)

print("\nSaved image copy report:")
print(DEMO_DATA_DIR / "image_copy_report.parquet")
print(DEMO_DATA_DIR / "image_copy_report.csv")


# ------------------------------------------------------------
# Add/refresh image_url columns in demo data
# ------------------------------------------------------------

def refresh_image_url_column(df, article_col="article_id"):
    if df is None or article_col not in df.columns:
        return df
    df = df.copy()
    df["article_id"] = df[article_col].map(normalize_article_id)
    df["image_url"] = df[article_col].map(article_id_to_frontend_url)
    return df


if "demo_articles" in globals() and "article_id" in demo_articles.columns:
    demo_articles = refresh_image_url_column(demo_articles, "article_id")
    demo_articles.to_parquet(DEMO_DATA_DIR / "demo_articles.parquet", index=False)
    demo_articles.to_csv(DEMO_DATA_DIR / "demo_articles.csv", index=False)
    print("Updated demo_articles with image_url")

if "demo_home_feed" in globals() and "article_id" in demo_home_feed.columns:
    demo_home_feed = refresh_image_url_column(demo_home_feed, "article_id")
    demo_home_feed.to_parquet(DEMO_DATA_DIR / "demo_home_feed.parquet", index=False)
    demo_home_feed.to_csv(DEMO_DATA_DIR / "demo_home_feed.csv", index=False)
    print("Updated demo_home_feed with image_url")

if "demo_similar_items" in globals():
    demo_similar_items = demo_similar_items.copy()

    if "similar_article_id" in demo_similar_items.columns:
        demo_similar_items["similar_image_url"] = demo_similar_items["similar_article_id"].map(article_id_to_frontend_url)

    if "query_article_id" in demo_similar_items.columns:
        demo_similar_items["query_image_url"] = demo_similar_items["query_article_id"].map(article_id_to_frontend_url)

    demo_similar_items.to_parquet(DEMO_DATA_DIR / "demo_similar_items.parquet", index=False)
    demo_similar_items.to_csv(DEMO_DATA_DIR / "demo_similar_items.csv", index=False)
    print("Updated demo_similar_items with image URLs")


# ------------------------------------------------------------
# Quick check: show copied folders
# ------------------------------------------------------------

created_image_folders = sorted([p.name for p in DEMO_PUBLIC_IMAGES_DIR.iterdir() if p.is_dir()])

print("\nDemo public images folder:")
print(DEMO_PUBLIC_IMAGES_DIR)

print("Created image subfolders sample:")
print(created_image_folders[:20])

HM_IMAGES_DIR: /kaggle/input/competitions/h-and-m-personalized-fashion-recommendations/images
Exists: True
Sample source image folders: ['010', '011', '012', '013', '014', '015', '016', '017', '018', '019']
Images selected for copy: 5000
Sample article_ids: ['0108775015', '0108775044', '0108775051', '0110065002', '0111565001', '0111565003', '0111586001', '0111593001', '0111609001', '0120129001']
Processed 500/5000 images...
Processed 1000/5000 images...
Processed 1500/5000 images...
Processed 2000/5000 images...
Processed 2500/5000 images...
Processed 3000/5000 images...
Processed 3500/5000 images...
Processed 4000/5000 images...
Processed 4500/5000 images...
Processed 5000/5000 images...

Image copy finished.
Copied: 4981
Already existed: 0
Missing: 19
Total processed: 5000

Saved image copy report:
/kaggle/working/hm-recommender/demo_export/data/image_copy_report.parquet
/kaggle/working/hm-recommender/demo_export/data/image_copy_report.csv
Updated demo_articles with image_url
Updated

## 12. Create frontend config

This file tells the React/FastAPI app where the exported data and images are located.


In [22]:
frontend_config = {
    "project_name": "H&M Fashion Recommender Demo",
    "final_model_name": demo_model_metrics.get("final_model_name"),
    "demo_export_dir": str(DEMO_EXPORT_DIR),
    "data_dir": str(DEMO_DATA_DIR),
    "public_dir": str(DEMO_PUBLIC_DIR),
    "public_images_dir": str(DEMO_PUBLIC_IMAGES_DIR),
    "frontend_image_base_url": "/images",
    "placeholder_image_url": "/images/placeholder-product.jpg",
    "files": {
        "customers": "data/demo_customers.json",
        "articles": "data/demo_articles.json",
        "home_feed": "data/demo_home_feed.json",
        "similar_items": "data/demo_similar_items.json",
        "similar_users": "data/demo_similar_users.json",
        "similar_user_taste_summary": "data/demo_similar_user_taste_summary.json",
        "model_metrics": "data/demo_model_metrics.json",
        "eda_summary": "data/demo_eda_summary.json",
        "image_copy_report": "data/image_copy_report.json",
    },
    "counts": {
        "demo_customers": int(len(demo_customers)),
        "demo_articles": int(len(demo_articles)),
        "home_feed_rows": int(len(demo_home_feed)),
        "similar_items_rows": int(len(demo_similar_items)),
        "similar_users_rows": int(len(demo_similar_users)),
        "images_attempted": int(len(image_copy_report)),
        "images_copied_or_existing": int(image_copy_report["status"].isin(["copied", "already_exists"]).sum()) if len(image_copy_report) > 0 else 0,
        "images_missing": int((image_copy_report["status"] == "missing_source").sum()) if len(image_copy_report) > 0 else 0,
    },
}

save_json(frontend_config, DEMO_DATA_DIR / "frontend_config.json")
display(pd.DataFrame([frontend_config["counts"]]))


Saved: /kaggle/working/hm-recommender/demo_export/data/frontend_config.json


,demo_customers,demo_articles,home_feed_rows,similar_items_rows,similar_users_rows,images_attempted,images_copied_or_existing,images_missing
0,150,5000,2000,10000,500,5000,4981,19


## 13. Final export summary


In [23]:
final_demo_export_summary = {
    "notebook": "09-demo-data-export.ipynb",
    "purpose": "Prepare web-ready demo data and copy only needed demo images.",
    "demo_export_dir": str(DEMO_EXPORT_DIR),
    "demo_data_dir": str(DEMO_DATA_DIR),
    "demo_public_images_dir": str(DEMO_PUBLIC_IMAGES_DIR),
    "hm_images_dir_found": str(HM_IMAGES_DIR) if HM_IMAGES_DIR is not None else None,
    "copy_demo_images": bool(COPY_DEMO_IMAGES),
    "counts": frontend_config["counts"],
    "data_files": sorted([p.name for p in DEMO_DATA_DIR.glob("*")]),
    "image_subfolders_created": sorted([p.name for p in DEMO_PUBLIC_IMAGES_DIR.iterdir() if p.is_dir()])[:50] if DEMO_PUBLIC_IMAGES_DIR.exists() else [],
}

print(json.dumps(final_demo_export_summary, indent=2, default=str))
save_json(final_demo_export_summary, DEMO_EXPORT_DIR / "demo_export_summary.json")

print("Demo data export completed successfully.")
print("Next step: copy demo_export/data and demo_export/public/images into the web app project.")


{
  "notebook": "09-demo-data-export.ipynb",
  "purpose": "Prepare web-ready demo data and copy only needed demo images.",
  "demo_export_dir": "/kaggle/working/hm-recommender/demo_export",
  "demo_data_dir": "/kaggle/working/hm-recommender/demo_export/data",
  "demo_public_images_dir": "/kaggle/working/hm-recommender/demo_export/public/images",
  "hm_images_dir_found": "/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations/images",
  "copy_demo_images": true,
  "counts": {
    "demo_customers": 150,
    "demo_articles": 5000,
    "home_feed_rows": 2000,
    "similar_items_rows": 10000,
    "similar_users_rows": 500,
    "images_attempted": 5000,
    "images_copied_or_existing": 4981,
    "images_missing": 19
  },
  "data_files": [
    "demo_articles.csv",
    "demo_articles.json",
    "demo_articles.parquet",
    "demo_customers.csv",
    "demo_customers.json",
    "demo_customers.parquet",
    "demo_eda_summary.json",
    "demo_home_feed.csv",
    "demo_home_feed.js